# Chapter 8: Stacks and Queues

The programs from Chapter 8 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## A Stack

`stack.py`

In [1]:
"""A stack: last in, first out. A Python list does all the work."""


class Stack:
    def __init__(self):
        self._items = []

    def push(self, value):
        self._items.append(value)  # the end of the list is the top

    def pop(self):
        if not self._items:
            raise IndexError("pop from an empty stack")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError("peek at an empty stack")
        return self._items[-1]

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

## Balanced Brackets

`brackets.py`

In [2]:
"""Are the brackets balanced? A stack remembers what must close next."""

PAIRS = {")": "(", "]": "[", "}": "{"}


def balanced(text):
    stack = []
    for ch in text:
        if ch in "([{":
            stack.append(ch)
        elif ch in PAIRS:
            if not stack or stack.pop() != PAIRS[ch]:
                return False  # a closer with no matching opener
    return not stack  # anything left open is unbalanced


for text in ["(a[b]{c})", "([)]", "((", "f(x) = [1, {2: 3}]", ""]:
    print(f"{text!r:>22}: {balanced(text)}")

           '(a[b]{c})': True
                '([)]': False
                  '((': False
  'f(x) = [1, {2: 3}]': True
                    '': True


## Monotonic Stacks

`monotonic.py`

In [3]:
"""A monotonic stack: for each day, how many days until it is warmer?

The stack holds the days still waiting for a warmer one, coldest on
top. Each new day pops every waiting day it is warmer than. Every day is
pushed and popped at most once, so the whole thing takes linear time.
"""


def days_until_warmer(temps):
    answer = [0] * len(temps)
    waiting = []  # indexes of days, temperatures decreasing upward
    for day, t in enumerate(temps):
        while waiting and temps[waiting[-1]] < t:
            earlier = waiting.pop()
            answer[earlier] = day - earlier
        waiting.append(day)
    return answer  # days never followed by a warmer one stay 0


temps = [21, 19, 18, 22, 20, 25, 24, 24]
print(temps)
print(days_until_warmer(temps))

[21, 19, 18, 22, 20, 25, 24, 24]
[3, 2, 1, 2, 1, 0, 0, 0]


## A Ring Buffer

`ring_queue.py`

In [4]:
"""A queue (first in, first out) in a ring buffer.

The items sit in fixed storage that wraps around: when the back reaches
the end, it continues at slot 0. Both ends move forward, so nothing is
ever shifted. When the storage is full, it doubles.
"""


class RingQueue:
    def __init__(self, capacity=8):
        self._data = [None] * capacity
        self._front = 0  # slot of the first item
        self._n = 0

    def __len__(self):
        return self._n

    def enqueue(self, value):
        if self._n == len(self._data):
            self._grow()
        back = (self._front + self._n) % len(self._data)
        self._data[back] = value
        self._n += 1

    def dequeue(self):
        if not self._n:
            raise IndexError("dequeue from an empty queue")
        value = self._data[self._front]
        self._data[self._front] = None
        self._front = (self._front + 1) % len(self._data)
        self._n -= 1
        return value

    def _grow(self):
        """Copy the items, in order, into storage twice the size."""
        old, size = self._data, len(self._data)
        self._data = [None] * (2 * size)
        for i in range(self._n):
            self._data[i] = old[(self._front + i) % size]
        self._front = 0

## A Queue from Two Stacks

`two_stack_queue.py`

In [5]:
"""A queue made of two stacks: an interview favorite.

New items go on the `inbox` stack. To take one out, pop from `outbox`;
when it is empty, pour the whole inbox into it, which reverses the
order so the oldest item is on top. Each item is moved at most once, so
the average cost per operation is constant.
"""


class TwoStackQueue:
    def __init__(self):
        self._inbox, self._outbox = [], []

    def __len__(self):
        return len(self._inbox) + len(self._outbox)

    def enqueue(self, value):
        self._inbox.append(value)

    def dequeue(self):
        if not self._outbox:
            while self._inbox:
                self._outbox.append(self._inbox.pop())
        if not self._outbox:
            raise IndexError("dequeue from an empty queue")
        return self._outbox.pop()

## Timing Four Queues

`bench_queues.py`

In [6]:
"""A queue of n items filled, then emptied: four implementations."""

from collections import deque

import bench
from ring_queue import RingQueue
from two_stack_queue import TwoStackQueue


def with_list(n):
    q = []
    for i in range(n):
        q.append(i)
    while q:
        q.pop(0)  # every remaining item shifts left


def with_deque(n):
    q = deque()
    for i in range(n):
        q.append(i)
    while q:
        q.popleft()


def with_class(make):
    def run(n):
        q = make()
        for i in range(n):
            q.enqueue(i)
        while len(q):
            q.dequeue()

    return run


sizes = [5_000, 10_000, 20_000, 40_000, 80_000]
results = {
    "list.pop(0)": bench.doubling(with_list, lambda n: (n,), sizes),
    "deque": bench.doubling(with_deque, lambda n: (n,), sizes),
    "RingQueue": bench.doubling(
        with_class(RingQueue), lambda n: (n,), sizes
    ),
    "TwoStackQueue": bench.doubling(
        with_class(TwoStackQueue), lambda n: (n,), sizes
    ),
}
bench.table(results)
bench.chart(results, "ch08_queues.png", "Fill and empty a queue")

         n     list.pop(0)           deque       RingQueue   TwoStackQueue
     5,000          886 us          215 us          1.6 ms          870 us
    10,000     6.5 ms x7.3     449 us x2.1     3.3 ms x2.1    1.76 ms x2.0
    20,000    31.7 ms x4.9     900 us x2.0    6.62 ms x2.0    3.51 ms x2.0
    40,000     125 ms x3.9    1.82 ms x2.0    13.1 ms x2.0    7.24 ms x2.1
    80,000     517 ms x4.1    4.04 ms x2.2    27.4 ms x2.1    14.4 ms x2.0
